In [ ]:
# splits_and_domains.py
import os
import glob
import cv2
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.cluster import MiniBatchKMeans

def extract_lab_domain_features(img_paths, sample_size=128):
    feats = []
    for p in img_paths:
        img = cv2.imread(str(p))
        if img is None:
            feats.append(np.zeros(6, dtype=np.float32))
            continue
        img_small = cv2.resize(img, (sample_size, sample_size), interpolation=cv2.INTER_AREA)
        lab = cv2.cvtColor(img_small, cv2.COLOR_BGR2LAB).astype(np.float32)
        mask = (img_small[:, :, 1] > 12)
        if np.any(mask):
            l_m, a_m, b_m = lab[:, :, 0][mask].mean(), lab[:, :, 1][mask].mean(), lab[:, :, 2][mask].mean()
            l_s, a_s, b_s = lab[:, :, 0][mask].std(), lab[:, :, 1][mask].std(), lab[:, :, 2][mask].std()
        else:
            l_m, a_m, b_m, l_s, a_s, b_s = 0.0, 0.0, 0.0, 1.0, 1.0, 1.0
        feats.append([l_m, a_m, b_m, l_s, a_s, b_s])
    return np.asarray(feats, dtype=np.float32)

def assign_pseudo_domains(df, n_domains=10, seed=42):
    print(f"Extracting CIE-LAB color/illumination features for {len(df)} images...")
    feats = extract_lab_domain_features(df["image_path"].values)
    kmeans = MiniBatchKMeans(n_clusters=n_domains, random_state=seed, batch_size=256, n_init=10)
    df["pseudo_domain"] = kmeans.fit_predict(feats)
    return df

def prepare_all_splits(data_root="./data", out_dir="./data_splits", seed=42):
    os.makedirs(out_dir, exist_ok=True)

    # 1. AIROGS Split (Train / Val / Test)
    airogs_csv = os.path.join(out_dir, "airogs_splits.csv")
    if os.path.exists(airogs_csv):
        df_air = pd.read_csv(airogs_csv)
        if "pseudo_domain" not in df_air.columns:
            df_air = assign_pseudo_domains(df_air, n_domains=10, seed=seed)
            df_air.to_csv(airogs_csv, index=False)
        print(f"[AIROGS] Loaded {len(df_air)} records -> {airogs_csv}")

    # 2. PAPILA External Validation CSV
    papila_csv = os.path.join(out_dir, "papila_external.csv")
    if os.path.exists(papila_csv):
        df_pap = pd.read_csv(papila_csv)
        print(f"[PAPILA] Verified {len(df_pap)} external validation records -> {papila_csv}")

    # 3. Chaksu External Validation CSV
    chaksu_csv = os.path.join(out_dir, "chaksu_external.csv")
    if os.path.exists(chaksu_csv):
        df_chk = pd.read_csv(chaksu_csv)
        print(f"[Chaksu] Verified {len(df_chk)} external validation records -> {chaksu_csv}")

if __name__ == "__main__":
    prepare_all_splits()